# Scraping des 250 discussions Kialo
Inspiré de https://github.com/4mbroise/ADC/blob/main/tool/kialo.ipynb

## Imports

In [ ]:
from enum import Enum
import requests
import pandas as pd
import os
import time

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from tqdm import tqdm

## Identifiants Kialo

In [ ]:
kialoUsername = "..."
secret        = "..."

## Options de tri et de filtre de l'API Kialo

In [ ]:
class KialoSort(Enum):
  RANK_ACTIVITY = "rank_and_latest_activity"
  VIEW          = "view_count"
  LAST_ACTIVITY = "latest_activity"

class KialoFilter(Enum):
  PROMOTED      = "promoted"
  PARTICIPATE   = "participate"
  LAST_ACTIVITY = "latest_activity"
  TAG           = "tag"
  TAG_ALL       = "tag_all"

## Outil d'accès à l'API Kialo

In [ ]:
class KialoTool:
  def getTags(self):
    tags = requests.get("https://www.kialo.com/api/v1/discussiontags")
    return [item['tagName'] for item in tags.json()["tags"]]

  def getDiscussions(self, filter: KialoFilter, sort: KialoSort, limit=3000):
    req = "https://www.kialo.com/api/v1/discussions?filter=" + str(filter.value) + "&sort=" + str(sort.value) + "&limit=" + str(limit) + "&skip=0"
    tags = requests.get(req)
    return tags.json()["discussions"]

  def discussions2urlID(self, discussions, export=False):
    idsUrl = [x["title"].lower().replace("?", "").replace(" ", "-")+"-"+str(x["id"]) for x in discussions]
    tags = [x["tags"] for x in discussions]
    if export:
      pd.DataFrame.from_dict({"kialoUrlId" : idsUrl, "tags" : tags}).to_csv("kialo-url-ids.csv")
    return idsUrl

k = KialoTool()

## Téléchargement des exports `.txt`
Kialo demande d'être connecté pour exporter un débat : on utilise Selenium pour se connecter puis ouvrir chaque lien d'export.

In [ ]:
def downloadDiscussions(disscusionUrlIds):
  path = os.path.abspath("rawData")
  os.makedirs(path, exist_ok=True)

  prefs = {"download.default_directory": path,
           "download.prompt_for_download": False,
           "download.directory_upgrade": True}
  options = Options()
  options.add_experimental_option("prefs", prefs)
  driver = webdriver.Chrome(options = options)

  driver.get("https://www.kialo.com/login")

  # Login
  id          = WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, "emailOrUsername")))
  password    = driver.find_element(By.ID, "password")
  loginButton = driver.find_element(By.CLASS_NAME, "login-form__submit")
  id.send_keys(kialoUsername)
  password.send_keys(secret)
  loginButton.click()

  # La page d'accueil a changé (redirection vers /my) : on attend de quitter /login
  WebDriverWait(driver, 30).until(lambda d: "/login" not in d.current_url)
  time.sleep(6)   # sans cette pause, le premier téléchargement échoue

  for urlId in tqdm(disscusionUrlIds):
    driver.get("https://www.kialo.com/export/" + urlId + ".txt")
    time.sleep(4)

  time.sleep(4)  # laisse le dernier téléchargement se terminer
  driver.quit()

## Étape 1 : les 250 discussions les plus actives et les mieux classées
Tri `rank_and_latest_activity`.

In [ ]:
discussions = k.getDiscussions(filter=KialoFilter.TAG, sort=KialoSort.RANK_ACTIVITY, limit=250)
downloadDiscussions(k.discussions2urlID(discussions, export=True))

## Étape 2 : suppression des débats non anglais
On utilise le champ `language` de l'API Kialo. Les modèles de la partie 2 sont entraînés sur de l'anglais.

In [ ]:
# On demande la langue de chaque débat à l'API (champ "language").
# L'en-tête du fichier ne suffit pas : Kialo écrit "Discussion Name:" en anglais
# pour certaines langues (slovaque, suédois, grec...).
for nom in os.listdir("rawData"):
    id_debat = nom.replace(".txt", "").split("-")[-1]
    reponse = requests.get("https://www.kialo.com/api/v1/discussions/" + id_debat)
    langue = reponse.json()["discussion"]["language"]
    if langue != "en":
        print("Supprimé :", nom, "(" + langue + ")")
        os.remove(os.path.join("rawData", nom))

print("Débats en anglais :", len(os.listdir("rawData")))

## Étape 3 : compléter jusqu'à 250 débats en anglais
On prend les débats anglais suivants dans le classement (champ `language` de l'API).

In [ ]:
ids_deja_la = []
for nom in os.listdir("rawData"):
    ids_deja_la.append(nom.replace(".txt", "").split("-")[-1])

discussions = k.getDiscussions(filter=KialoFilter.TAG, sort=KialoSort.RANK_ACTIVITY, limit=400)

manquants = []
for d in discussions:
    if d["language"] == "en" and str(d["id"]) not in ids_deja_la:
        manquants.append(d)

manquants = manquants[:250 - len(ids_deja_la)]
for d in manquants:
    print(d["title"])

downloadDiscussions(k.discussions2urlID(manquants))
print("Fichiers dans rawData :", len(os.listdir("rawData")))

## Étape 4 : fichier des tags
On récupère les tags de chaque débat via l'API, à partir de l'id qui termine le nom du fichier.

In [ ]:
lignes = []
for nom in sorted(os.listdir("rawData")):
    url_id = nom.replace(".txt", "")
    id_debat = url_id.split("-")[-1]
    reponse = requests.get("https://www.kialo.com/api/v1/discussions/" + id_debat)
    tags = reponse.json()["discussion"]["tags"]
    lignes.append({"kialoUrlId": url_id, "tags": tags})

pd.DataFrame(lignes).to_csv("kialo-url-ids.csv")
print("Débats dans le CSV :", len(lignes))